## 网络 · 块0 总纲：五层模型与封装 / 解封装

> 2026-10-11 · 来源：学习对话

- **我的理解**（原话）：「开始计算机网络」。前置背景：此前网络是**零散触达**——从 socket 视角切入，先跳到了应用层（带过）与传输层块1（已展开），**从未立过总纲地图**；本轮补总纲，给已学内容建立坐标系。
- **机制层（①）**：
  - 五层职责 / PDU / 设备标识：
    - 应用层 · 面向具体应用 · 报文 message · （端系统，无专属设备）
    - 传输层 · 进程↔进程、端到端 · 报文段 / 用户数据报 · 端口号
    - 网络层 · 主机↔主机、选路 · 分组 / 数据报 · 路由器、IP 地址
    - 链路层 · 相邻结点、成帧与差错 · 帧 frame · 交换机 / 网桥、MAC 地址
    - 物理层 · 比特流传输 · 比特 bit · 中继器 / 集线器
  - 记忆链：应用层管「干什么」→ 传输层管「哪个进程」→ 网络层管「哪台机器」→ 链路层管「下一跳是谁」→ 物理层管「怎么变成信号」；负责范围**逐级缩小**：进程 → 主机 → 相邻结点 → 比特。
  - **封装**：发送方自顶向下，每层在**首部**加控制信息（链路层还在**尾部**加 FCS 校验）；**解封装**：接收方自下而上，逐层剥首部上交。
  - 首部是**层层叠加**而非替换（以太网帧内自外向内：帧首部 / IP 首部 / TCP 首部 / 数据）；**交换机只看帧首部、路由器只看 IP 首部**，不碰内层——这正是分层的收益。
  - **socket 位置**：夹在应用层与传输层之间，是**编程接口**而非一层。`SOCK_STREAM` = TCP、`SOCK_DGRAM` = UDP；`listen` / `accept` 背后的半连接 / 全连接队列对应三次握手。
- **设计层（②）**：
  - **为什么分层**：把「端到端可靠性 / 选路 / 相邻结点可靠性 / 信号」四类问题**解耦**，各层独立演进、独立复用（上层不管下层怎么实现）。
  - **为什么是五层而非 OSI 七层**：OSI 的会话层 / 表示层在 TCP/IP 里并入应用层——工程上无独立价值（会话管理谁用谁自己做，表示转换由编解码器承担）。
  - **现实对照**：Linux 网络栈（`socket` → `tcp/ip` → `netdev` → 驱动）。**边界**：408 考模型、封装、各层职责，不考内核实现。
  - **socket 的边界（重要）**：socket 能锚定**传输层**直觉（端口 / 复用分用），**看不到网络层**（仅 IP 地址间接可见）——它是入口，不是主线。这正是"从 socket 切入会先摸到传输层、绕不到网络层"的原因。
- **考试侧重点**：① 判定「某功能 / 协议 / 设备 / PDU 名属于哪一层」；② 写出某次通信自顶向下的封装过程（或自下而上的解封装）；③ 五层 vs OSI 七层的映射（会话层 / 表示层并入应用层）。
- **易错点**：把**设备**归错层（交换机 = 链路层、路由器 = 网络层、集线器 / 中继器 = 物理层）；PDU 名称混用（报文 / 报文段 / 分组 / 帧）；以为首部是"替换"而非"叠加"。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-12


## 网络 · 大观：五层各自在对付什么「不靠谱」

> 2026-10-12 · 来源：学习对话

- **我的理解**（原话节选）：「传输层就是提供了进程 port 端到端之间的通信，对每个进程端口到进程端口的连接进行一个管理，当然 udp 本身不维持这个连接，只是说 tcp udp 都有一条稳定的独立通道；网络层则是网络设备之间的，由 IP 设定……数据链路层，则是对物理层的封装，将 01 信号封装成 packet……也就是包的检错纠错，链路的拥堵等等；物理层就是 01 信号编码相关的问题以及时延的计算相关的物理特性问题」。
- **核心澄清 1：每层对付的「不靠谱」不同，范围不同**

| 层 | 范围 | 失效模式 | 对策 |
|---|---|---|---|
| 传输层 | 进程↔进程（端到端） | 丢失 / 乱序 / 重复 | 确认 + 重传 + 拥塞控制 |
| 网络层 | 主机↔主机（整条路径） | 不可达 / 环路 / 规模 | **尽力而为**（刻意不保证）+ 路由聚合 |
| 链路层 | 相邻结点（一段链路） | 比特差错 / 信道争用 | CRC 检错（+ 可选重传 / CSMA-CD） |
| 物理层 | 一段介质 | 衰减 / 噪声 / 时延 | 编码 + 调制 |

- **核心澄清 2：逐段可靠 ≠ 端到端可靠（端到端原则）**
  - 链路层的可靠是**逐段（hop-by-hop）**：A→R1、R1→R2、R2→B 各自保证，范围 = 一段链路。
  - 传输层的可靠是**端到端（end-to-end）**：只在 A、B 之间保证，范围 = 全程。
  - **逐段为什么替代不了端到端**：ⓐ 中间路由器缓存满会**丢包**（包在链路上是正确收到的，链路层管不着后续）；ⓑ 中间设备会崩、**路径会变**；ⓒ 逐段只承诺「交给下一跳」，不承诺「交到终点」。
  - **结论（端到端原则）**：某功能若只有端点能正确实现，就不该放进中间设备 → 所以 **IP 刻意尽力而为，可靠性交给 TCP**。
  - **反过来，逐段检错还有必要吗**：有——在**高误码链路**（无线、老式电话线）上本地修复一个比特错，可避免端到端重传整条路径，省带宽降时延。
  - **以太网的选择**：有线链路误码率极低，逐段重传收益盖不过开销，故**只检错不重传**（CRC 不过即丢弃），把重传推给上层 TCP。**这是权衡，不是缺陷**（408 常问「以太网为何不提供可靠传输」）。
  - **补充**：链路层的可靠传输（GBN / SR 滑动窗口）是**可选服务**——HDLC、PPP 的可靠版本、无线链路会开；以太网选择不开。
- **核心澄清 3：三个易混的「争用 / 流量 / 拥塞」——判据是「它在保护谁」**

| 机制 | 保护谁 | 属于 | 典型 |
|---|---|---|---|
| 介质访问控制 | 一条共享信道（谁先发） | 链路层 | CSMA/CD、CSMA/CA |
| 流量控制 | **接收方**（别冲爆我的缓存） | 传输层 | 滑动窗口 / 接收窗口 rwnd |
| 拥塞控制 | **网络中间**（别压垮路由器和链路） | 传输层（网络层辅助） | 慢开始、拥塞避免、快重传 / 快恢复 |

  - 分界线：**拥塞 = 网络内部过载（端到端视角）；争用 = 一段共享信道上的抢夺（本地视角）**。原话「链路的拥堵」准确名字是**介质访问控制**，不是拥塞控制。
  - 发送窗口 = min(接收窗口 rwnd, 拥塞窗口 cwnd)：前者怕压垮**对方**，后者怕压垮**网络**。
- **术语纠正（易错）**：原话说「链路层把 01 信号封装成 packet」——**链路层 PDU 是「帧 frame」**；「分组 / packet」是网络层的。408 送分点。
- **考试侧重点**：① 「为什么可靠性要在多层实现 / 逐段与端到端的区别」（理解题）；② 流量控制 vs 拥塞控制 vs 介质访问控制 的辨析；③ PDU 名称与层级的对应。
- **易错点**：把链路层说成分组 / packet；以为链路层有拥塞控制；以为「逐段可靠」能替代「端到端可靠」。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## 网络 · 检错 vs 纠错：链路层只「检」不「修」

> 2026-10-12 · 来源：学习对话

- **我的理解**（原话节选）：「既不能把错误的包上传到网络层, 也要保证少数 bit 错误时的纠错, 因为如果不稳定导致万分之一的错误率也不能把大部分包全部重传, 所以加上简单的检错纠错就是中间的最优解」。
- **判定：前半句对，后半句是术语混用。** 链路层做的是**检错（detection）**，**不做纠错（correction）**。
- **检错 vs 纠错**（408 送分点）：

| | 检错 detection | 纠错 correction |
|---|---|---|
| 能力 | 只回答「错了吗」 | 回答「错了吗 + 错在哪」 |
| 代表 | CRC、校验和 | 海明码、FEC |
| 冗余开销 | 少 | 多（要定位就要更多冗余位） |
| 出错后 | **丢弃** / 交上层重传 | **直接改正** |
| 408 归属 | 链路层（CRC）/ 传输层、网络层（校验和） | **01-数据表示与编码**（海明码）——**不在链路层考** |

- **CRC 只能检不能纠**：生成多项式做模 2 除法，余数为 0 则**认为**无错（有极小漏检率），非 0 则丢弃；它**不指出哪一位错了**。
- **「不污染上层」是检错的第一价值**（原话这句对）：坏帧若在链路层不放行，就不会被上层当成正确数据一路算下去。
- **为什么链路层不纠错**：纠错码冗余位多、开销大；而链路层只管一段，**丢弃让端到端重传**（低误码率下极少发生）比逐段纠错划算。**除无线链路用 FEC 外，有线链路一律「只检不修」。**
- **全栈一致的思路**：链路层 CRC、UDP 校验和、IPv4 首部校验和——**都是只检不修（发现错就丢）**；真正的"修"（重传）交给唯一能负责端到端的那一层（TCP）。
- **数量级纠正**：原话说「万分之一」（10^-4）——有线以太网要求 BER 约 10^-10 量级，低了好几个数量级。**有线链路的低误码率由物理层保证，CRC 只是最后一道保险。**
- **考试侧重点**：① 检错 vs 纠错的辨析及各自代表码；② CRC 的计算（模 2 除法，见块4 展开）；③ 海明码的归属（01，不在链路层）。
- **易错点**：把 CRC 说成「能纠错」；把海明码归到链路层；以为网络层会校验数据部分。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## 网络 · 海明码用在哪：内存 ECC，不是网络传输

> 2026-10-12 · 来源：学习对话（用户提问）

- **我的提问**（原话）：「那海明码用在哪? 现代网络质量不需要纠错吗? 用在无线通信吗」。
- **核心结论 1：海明码的主战场是「存储器 ECC」，不是网络传输。**
  - 服务器 / 工作站内存用 **ECC 内存**（Error-Correcting Code memory），常见 **SECDED**：纠 1 位、检 2 位。
  - 为什么内存必须**纠**而不能只**检**：CPU 读内存**没有「重传」这个选项**——读到的就是它读到的。DRAM 会因**宇宙射线 / 电磁干扰**发生**软错误**（soft error，电容电荷翻转），必须当场修正。
  - 所以 **408 把海明码放在 `01-数据表示与编码`**（存储器校验语境），**不放在链路层**——上一轮「链路层用 CRC 不用海明码」的原因正在这里：**内存没有重传，链路有重传。**
- **核心结论 2：现代网络仍然需要纠错，但「谁纠、怎么纠」按介质分工（判据是误码率）。**

| | 有线以太网 | 无线 / 卫星 |
|---|---|---|
| 误码率 BER | 约 10^-10 | 约 10^-3 ~ 10^-6（高出 4~7 个数量级） |
| 策略 | **只检不修**（CRC 丢帧） | **FEC 前向纠错为主** |
| 重传 | 交给 TCP | 纠错失败再靠 CRC + ARQ 兜底 |
| 纠错码 | 不用 | LDPC / Turbo / Polar |

- **核心结论 3：无线确实靠纠错活着，但主力不是海明码本身。**
  - 无线误码率高（信号衰减、多径衰落、干扰、移动引起信道快速变化）。**这种误码率下若只检错 + 重传，重传次数会爆炸、吞吐崩溃**，故必须**前向纠错（FEC）**——接收方不靠反馈，自行改正。
  - 现代无线主力码：**LDPC**（WiFi 6/7、5G 数据信道）、**Turbo 码**（3G/4G）、**Polar 码**（5G 控制信道）。海明码**本身**纠错能力弱（只纠 1 位），现代无线基本不用。
  - **思想一脉相承**：都是「加冗余位 → 接收方自行纠正」；海明码（Hamming, 1950）是这条路的源头，不是当代主力。
  - **现代无线标准组合「FEC + CRC + ARQ」**（HARQ，混合自动重传请求）：FEC 先纠，CRC 验残余错误，ARQ 最后兜底。**408 不考。**
- **三问答案串起来**：
  - 海明码用在哪 → **内存 ECC**（服务器内存 SECDED），不是网络传输；
  - 现代网络需要纠错吗 → **需要**：有线靠物理层压低误码率故不纠，**无线误码率天生高，必须纠**；
  - 用在无线吗 → **思想用了，码不是它**（主力是 LDPC / Turbo / Polar）。
- **易错点**：以为海明码是网络（链路层）的纠错码（**它是存储器 ECC 的经典码**）；以为无线用海明码纠错（**用 LDPC / Turbo / Polar**）；以为「现代网络质量好所以不需要纠错」（有线的低误码率靠物理层保证，无线的误码率天生高得多）。
- **考试侧重点**：① 海明码编码原理与冗余位关系（`2^k >= n + k + 1`）→ 归 `01-数据表示与编码`；② 链路层差错控制只考 **CRC 计算** 与「检错 vs 纠错」的定性辨析；③ **不考** LDPC / Turbo / Polar / 交织 / HARQ。
- **边界**：本 cell 属链路层语境的「差错控制全景」；海明码编码原理本身待 `01-数据表示与编码.ipynb` 建立后归档。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## [CS144①] 设计理念：C++ 为什么能替换 Linux 网络协议栈

> 2026-10-20 · 来源：CS144 项目重修（learning-408/cs144 实读 + check0–7 官方 PDF）

- **我的理解**（本条由代码 + 官方 PDF 推导，非对话原话）：替换不是劫持内核，而是**重建** —— 你的 TCP/IP 就是一个普通用户态进程。三条支撑事实：

  1. **唯一的契约是文件描述符**：`Socket` / `TunFD` / `TapFD` / stdin 全都继承自 `FileDescriptor`（`cs144/util/file_descriptor.hh`，引用计数持有 fd）。整个 Minnow 里数据流动只有 `read()` 和 `write()`。

  2. **每层的形状被压成同一种**：一个 struct + 一对 `parse(Parser&)` / `serialize(Serializer&)`，报文本身是 `vector<string>`（`cs144/util/parser.hh`）。剥一层头就 `remove_prefix`，下一层接着剥。

  3. **时间只有两个来源**：`EventLoop`（poll 封装，真实事件）与 `tick(ms)`（协议内虚拟时间，check3 明文禁止碰真实时钟）。

- **出站方式的三个等级**（check5 官方 PDF §0 原文列举）：

  1. **TCP-in-UDP-in-IP** —— 用 `UDPSocket`，内核写 UDP/IP/以太头，你只给 payload 和目的地址；

  2. **TCP-in-IP** —— 用 **TUN 设备**，你写完整 IP 头，内核写以太头并真发；

  3. **TCP-in-IP-in-Ethernet** —— 用 **TAP** 或 `AF_PACKET`，连以太头和 ARP 都自己写。

- **一句话串起来**：每往后做一个 checkpoint，就把「内核帮忙写的那部分头」拿走一层变成自己写。

- **项目映射**（出处：`cs144/util/tun.cc`、`cs144/scripts/tun.sh`、`cs144/apps/tcp_ipv4.cc`）：`open('/dev/net/tun')` + `ioctl(TUNSETIFF, IFF_TUN|IFF_NO_PI)` 把进程 fd 与虚拟网卡焊死；`tun.sh` 再补三件事 —— 给 `169.254.144.1/24`（link-local 段）、`iptables ... MASQUERADE`（NAT）、`ip_forward=1`。**最后一公里（以太网 + 出物理网卡 + NAT）仍是内核干的**：内核在这里是被当成传输介质，不是被替换掉。

- **真正的替换点在 socket 层**（`cs144/util/tcp_minnow_socket.hh`）：`TCPMinnowSocket` 继承 `LocalStreamSocket`（它仍然是 Socket、仍然是 fd），内部另起一个线程跑 `TCPPeer` + `EventLoop`。官方注释原话大意是：「另一个线程负责本来由内核替 TCPSocket 做的那些后端活」。于是 check4 的全部改动只有一行：把 `webget.cc` 里的 `TCPSocket` 换成 `CS144TCPSocket`。

- **考试侧重点**：① 分层与封装（每层各加谁的头）；② IPv4 首部字段的现实含义（TTL / 首部校验和 / 协议字段 6=TCP）；③ NAT 与私有地址的现实后果。

- **易错点**：以为「替换」等于「内核彻底不参与」 —— 错。`NetworkInterface` 之前的链路层一跳一直是 Linux 代劳的（见 check5 才会自己写以太头 + ARP）；以为 cs144 里有拥塞控制 —— 没有，那里的窗口就是纯 rwnd。

- **待解决问题**：这条线索在 check5（以太头自己写）之后值得再回看一次。

- **复习计划**：R1 @ 2026-10-21 · R2 @ 2026-10-23 · R3 @ 2026-10-27


## [CS144②] 网络接口与 ARP（check5）

> 2026-10-20 · 来源：CS144 项目重修（learning-408/cs144 实读 + check0–7 官方 PDF）

- **我的理解**（Evidence-based）：`NetworkInterface` 是 IP 世界与以太网世界之间的唯一翻译官，**同一份代码被复用两次** —— 作为主机 TCP/IP 栈的最底层，也作为路由器的一块网卡（check5 官方图 1 左右两侧）。

- **项目映射**（出处：`cs144` 分支 `origin/check7-startercode:src/network_interface.hh`）：三个方法 `send_datagram(dgram, next_hop)` / `recv_frame(frame)` / `tick(ms)`；发送靠虚函数口 `OutputPort::transmit()`（依赖注入，单测可插一个假的网口）；学到的 IP→MAC 映射只留 **30 秒**；同一个 IP 的 ARP request **5 秒内不重发**。

- **核心机制**：未知下一跳 MAC → 广播 ARP request + **把数据报排队**；reply 与 request 里的 sender 信息都顺手学（30 s）；目标 MAC 既不是广播也不是自己的帧一律忽略；IPv4 帧解析成功才塞进 `datagrams_received()`。

- **为什么值得做**：这三个动作就是课本上「MAC 地址 + ARP」那几页的实现版，做完以后「有了 IP 为什么还要 MAC 地址」这道题就不是背出来的了。

- **考试侧重点**：★链路层必考。以太网帧格式、MAC 寻址、ARP 三步（广播请求 / 单播响应 / 缓存）、以及 ARP 缓存带来的首次延迟。

- **易错点**：① **把最终目的地当成下一跳**（check5 PDF 用脚注专门警告，全 lab 第一号语义错误）；② 5 秒抑制是「同一 IP」，不是全局限流；③ request 和 reply 里的 sender 信息都要缓存；④ 只有 target IP 等于自己的 IP 才回 reply。

- **边界**：真实系统还会做 gratuitous ARP、重复地址检测、ARP 长时间无回应后回 ICMP host unreachable（官方明说 lab 不需要做）。这几条 408 不考。

- **复习计划**：R1 @ 2026-10-21 · R2 @ 2026-10-23 · R3 @ 2026-10-27


## [CS144·验收] 真题对位变式：GBN/SR 窗口与最小帧长（§8 题型 8–9）

> 2026-10-22 · 来源：CS144 真题对位变式（自编，题目与解答真源 = `modules/408/CS144.md` §8.3；全部答案先手算复核再落盘）

- **用法（对齐 LEARNING.md §六）**：限时作答——选择 2 分钟、大题 15–20 分钟；先遮住【解】自己算，再对答案。卡住的题回 `modules/408/CS144.md` §3.1 / §3.5 / §2.8 回补，并按归因维度记账：概念不清 / 计算失误 / 没记住 / 审题错 / 项目误导 / 存疑待核。

**题型 8：GBN / SR 窗口上限与 32 位序号**（★核心 · check1 / check2 对位）

【题】(1) 序号字段 3 位（0–7）：GBN 与 SR 的发送窗口上限各多少？SR 两窗口满足什么关系？
(2) TCP 序号 32 位、窗口字段只有 16 位（≤ 65535）：结合 CS144 说明为什么工程上不担心「序号回绕导致新旧分组混淆」？
(3) check1 的 Reassembler 里「接收窗口的内存上限」公式是什么？

【解】(1) GBN：$W \le 2^3 - 1 = 7$（留一个序号当分界）；SR：$W \le 2^{3-1} = 4$，约束 $W_{\text{发送}} + W_{\text{接收}} \le 2^n$（4 + 4 = 8）。
(2) 三重护栏：**ISN 随机**（新连接序号区间与旧连接错开，check2 官方给的理由）、**窗口 ≤ 65535 远小于 $2^{31}$**（一个窗口内的序号跨度不会绕进重叠区）、**分组寿命有限（MSL）**；check2 的 unwrap 取「离 checkpoint 最近」正是在窗口跨度 < $2^{31}$ 的前提下消歧。
(3) first_unacceptable = bytes_pushed + available_capacity = capacity + bytes_popped——重组器内部（红）+ ByteStream 缓冲（绿）两块合计不超过 capacity。

**题型 9：最小帧长与 lab 边界**（☆常规 · 「现实边界」怎么用）

【题】CS144 的 EthernetFrame 不做 FCS 校验、不做 64 字节最小帧长填充，照样跑通全部测试。
(1) 真实以太网 64 字节下限是为谁服务的？
(2) 为什么 CS144 可以不做？
(3) 这条「lab 缺席」对 408 意味着什么？

【解】(1) 为 **CSMA/CD 冲突检测**服务：帧太短会在检测到冲突前发完，发送方误判成功——64B 下限的来历 = 争用期内必须还在发送（512 bit 时间），它**不是帧语义的一部分**。
(2) CS144 的链路是虚拟网口/交换式全双工链路，**没有冲突域**，CSMA/CD 整体缺席；FCS 由真实网卡硬件处理，教学栈只管「形状对」。
(3) **lab 缺席 ≠ 考点缺席**：408 照考 64B 的来历与争用期计算——这正是 `CS144.md` §2.8「现实边界清单」的用法：每条 lab 简化都要能反向说出「课本考什么」。

- **复习计划**：R1 @ 2026-10-23 · R2 @ 2026-10-25 · R3 @ 2026-10-29